In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder, PowerTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.feature_selection import SelectKBest, chi2, f_classif, RFE, MutualInfoClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer

In [7]:
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('ggplot')
sns.set_palette("husl")

# Define Column Names for NSL-KDD (Standard List)
COL_NAMES = ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label", "difficulty_level"]
# Load Data (Assuming data is in ../SentinelNet/data or ../data relative to notebook)
# You might need to adjust the path based on your exact folder structure
train_path = './KDDTrain+.txt'
test_path = './KDDTest+.txt'

# Try loading with error handling for paths
try:
    df_train = pd.read_csv(train_path, header=None, names=COL_NAMES)
    df_test = pd.read_csv(test_path, header=None, names=COL_NAMES)
    print(f"Training Data Shape: {df_train.head()}")
    print(f"Test Data Shape: {df_test.tail()}")
except FileNotFoundError:
    print("Files not found. Please check the paths. Trying absolute paths or checking local directory...")
    # Fallback to local directory if paths are different
    if os.path.exists('KDDTrain+.txt'):
        df_train = pd.read_csv('KDDTrain+.txt', header=None, names=COL_NAMES)
        df_test = pd.read_csv('KDDTest+.txt', header=None, names=COL_NAMES)

# Combine for initial analysis (optional, but good for understanding full global distributi

Training Data Shape:    duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                         0

In [10]:
exclude_cols = ['label', 'attack_class', 'difficulty_level', 'is_outlier']
print(exclude_cols)

['label', 'attack_class', 'difficulty_level', 'is_outlier']


In [11]:
# Target variable
y = df_train['label']

# Feature columns
X = df_train.drop(columns=['label'])

In [12]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()

y_encoded = labelencoder.fit_transform(y)


In [14]:
# Separate input features and target column
target_col = 'label'

feature_data = df_train.drop(columns=[target_col])
target_data = df_train[target_col]

In [16]:
cat_cols = X.select_dtypes(include=['object']).columns

print("Categorical Columns:", list(cat_cols))

Categorical Columns: ['protocol_type', 'service', 'flag']


In [17]:
from sklearn.preprocessing import LabelEncoder

# Identify text-based columns
text_columns = feature_data.select_dtypes(include='object').columns

encoder_map = {}

for column in text_columns:
    
    encoder = LabelEncoder()
    
    feature_data[column] = encoder.fit_transform(feature_data[column])
    
    encoder_map[column] = encoder

In [20]:
target_encoder = LabelEncoder()

encoded_target = target_encoder.fit_transform(target_data)

In [21]:
from sklearn.model_selection import train_test_split

train_features, test_features, train_labels, test_labels = train_test_split(
    feature_data,
    encoded_target,
    test_size=0.25,
    random_state=7,
    stratify=encoded_target
)

In [27]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Combine train + test labels
all_labels = pd.concat([df_train['label'], df_test['label']])

# Fit on all labels
le.fit(all_labels)

# Transform separately
df_train['label'] = le.transform(df_train['label'])
df_test['label'] = le.transform(df_test['label'])

TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

In [28]:
attack_mapping = {
    'normal': 'normal',
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS',
    'satan': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'warezmaster': 'R2L', 'warezclient': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R'
}

df_train['attack_class'] = df_train['label'].map(attack_mapping)
df_test['attack_class'] = df_test['label'].map(attack_mapping)

# Encode
le = LabelEncoder()
all_classes = pd.concat([df_train['attack_class'], df_test['attack_class']])
le.fit(all_classes)

y_train = le.transform(df_train['attack_class'])
y_test = le.transform(df_test['attack_class'])

In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Predictions
y_pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("===== MODEL EVALUATION =====")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

NameError: name 'model' is not defined